# QuickCart Project Dark Matter
## Reproducible network design analysis

This notebook is the analysis companion to `final_report.docx` and `executive_deck.pptx`. It documents the metric construction, radius rule, counterfactual assignment, and outputs.

### Executive result
Open CS-01 (Pune), CS-07 (Hyderabad), and CS-11 (Jaipur); close PUN-02; relocate PUN-07 to CS-01. The portfolio estimate lowers C2S while increasing catchment coverage.

In [1]:
from pathlib import Path
import pandas as pd
OUT=Path('.')
metrics=pd.read_csv(OUT/'before_after_portfolio_metrics.csv')
metrics

,city,scenario,orders,active_facilities,new_sites,closures,avg_cost_to_serve,variable_cost_per_order,fixed_cost_per_order,avg_distance_km,breach_rate,on_time_rate,coverage_rate,monthly_fixed_total_inr,fitout_total_inr
0,All cities,Current baseline,186156,18,0,PUN-02 + PUN-07,71.743264,52.878441,18.864823,2.211470,0.246605,0.753395,0.726842,1755900.0,0.0
1,All cities,Recommended network,186156,19,3,PUN-02 + PUN-07,69.777103,48.953708,20.823395,1.926817,0.182200,0.817800,0.805652,1938200.0,4470000.0


### Definitions and assumptions

1. Haversine distance, one way, order by order.
2. Breach is actual delivery time greater than the 10-minute promise.
3. Fixed cost is allocated by store and calendar month.
4. Radius is the first 0.5 km ring with breach rate above 20%; orders beyond it are outside catchment.
5. Candidate fixed cost is city median existing monthly fixed cost; candidate radius is city median observed radius.
6. After-network breach is an expected probability from a city-specific logistic model on distance; this prevents claiming unobserved future actual delivery times.

In [2]:
radii=pd.read_csv(OUT/'serviceable_radii.csv') if (OUT/'serviceable_radii.csv').exists() else pd.DataFrame()
print('Radii and trigger rings')
radii

Radii and trigger rings


,store_id,serviceable_radius_km,first_breach_ring_lower_km,first_breach_rate,city,store_pincode,monthly_rent_inr,monthly_fixed_cost_inr
0,HYD-01,3.5,3.0,0.293515,Hyderabad,500081,55200,66700
1,HYD-02,3.0,2.5,0.223058,Hyderabad,500034,58300,64700
2,HYD-03,2.5,2.0,0.212963,Hyderabad,500016,44600,58700
3,HYD-04,3.0,2.5,0.229263,Hyderabad,500084,47100,57400
4,HYD-05,3.0,2.5,0.433333,Hyderabad,500089,39700,52100
5,HYD-06,3.0,2.5,0.262832,Hyderabad,500072,43400,60100
6,JAI-01,3.5,3.0,0.367992,Jaipur,302015,33500,48200
7,JAI-02,4.0,3.5,0.622549,Jaipur,302017,31600,46900
8,JAI-03,3.0,2.5,0.337719,Jaipur,302004,36000,50200
9,JAI-04,4.0,3.5,0.736364,Jaipur,302018,29100,44900


In [3]:
print('City metrics')
pd.read_csv(OUT/'before_after_city_metrics.csv')

City metrics


,city,scenario,orders,active_facilities,new_sites,closures,avg_cost_to_serve,variable_cost_per_order,fixed_cost_per_order,avg_distance_km,breach_rate,on_time_rate,coverage_rate,monthly_fixed_total_inr,fitout_total_inr
0,Hyderabad,Current baseline,68583,6,0,NaN,71.309045,52.412234,18.896811,2.157986,0.243544,0.756456,0.677573,648000.0,0.0
1,Jaipur,Current baseline,38526,5,0,NaN,74.723888,54.514159,20.209728,2.274577,0.286404,0.713596,0.725510,389300.0,0.0
2,Pune,Current baseline,79047,7,0,NaN,70.667303,52.485715,18.181588,2.227117,0.229863,0.770137,0.770238,718600.0,0.0
3,Pune,Recommended network,79047,6,1,PUN-02|PUN-07,66.335216,49.595808,16.739408,2.027155,0.180191,0.819809,0.829582,661600.0,1650000.0
4,Hyderabad,Recommended network,68583,7,1,NaN,70.959653,48.085180,22.874473,1.833984,0.174910,0.825090,0.767129,784400.0,1580000.0
5,Jaipur,Recommended network,38526,6,1,NaN,74.733965,49.182389,25.551576,1.886204,0.199299,0.800701,0.825131,492200.0,1240000.0


In [4]:
print('Recommended facility loads')
pd.read_csv(OUT/'recommended_facility_loads.csv')

Recommended facility loads


,city,assigned_facility,orders,avg_distance_km,p90_distance_km,expected_breach_rate,coverage,avg_after_cost,avg_fixed_cost
0,Hyderabad,HYD-04,14874,2.445353,4.086387,0.291181,0.591300,69.904263,14.051365
1,Hyderabad,CS-07,11192,1.675230,3.637396,0.149730,0.796015,70.593075,24.374553
2,Hyderabad,HYD-06,10043,1.238152,3.376310,0.081136,0.877327,61.712505,20.611371
3,Hyderabad,HYD-01,9550,1.595935,2.847833,0.090353,0.964188,69.410924,25.528796
4,Hyderabad,HYD-02,9062,2.600261,6.424459,0.315570,0.659678,84.815256,27.146325
5,Hyderabad,HYD-03,8439,1.617296,3.494104,0.144264,0.680057,70.130551,24.481574
6,Hyderabad,HYD-05,5423,1.064152,1.452136,0.043182,0.953716,72.600333,33.855799
7,Jaipur,JAI-03,8586,1.702925,2.684941,0.095821,0.978919,64.874319,20.079199
8,Jaipur,CS-11,6947,1.832090,3.936903,0.207312,0.826544,78.668277,29.624298
9,Jaipur,JAI-02,6565,2.319842,5.291499,0.323185,0.686672,79.849146,23.914699


In [5]:
print('Priority demand pockets')
pd.read_csv(OUT/'priority_demand_pockets.csv').head(20)

Priority demand pockets


,city,delivery_pincode,orders,population_density_per_sqkm,estimated_population,current_breach_rate,current_coverage,uncovered_orders
0,Hyderabad,500049,7946,17800,119260,0.492701,0.348226,5179.0
1,Hyderabad,500032,6659,11200,98560,0.418231,0.415828,3890.0
2,Hyderabad,500019,6325,14700,107310,0.655494,0.112569,5613.0
3,Hyderabad,500018,5305,15100,77010,0.506126,0.054854,5014.0
4,Hyderabad,500001,2201,24800,79360,0.993639,0.000000,2201.0
5,Jaipur,302012,4485,15800,94800,0.506800,0.424526,2581.0
6,Jaipur,302019,3152,11400,75240,0.824239,0.131345,2738.0
7,Jaipur,302020,2623,10900,85020,0.936332,0.003812,2613.0
8,Jaipur,302021,2503,13700,69870,0.909708,0.000000,2503.0
9,Pune,411057,8965,22600,144640,0.316899,0.682320,2848.0


### Optimization audit
Every non-empty existing/candidate subset is enumerated at exact order level. The balanced guardrail requires at least a 2 percentage-point improvement in both expected breach and coverage in each city, then minimizes expected cost-to-serve. The selected network ranks first in each city. See `optimization_summary.csv`, `exhaustive_exact_scenarios.csv`, `optimization_method.md`, and `images/optimization_frontier.png`.

In [6]:
print('Optimization summary')
pd.read_csv(OUT/'optimization_summary.csv')

Optimization summary


,city,enumerated_subsets,valid_scenarios,guardrail_scenarios,selected_network,selected_rank_by_cost,selected_rank_by_economic_cost,selected_avg_cost,selected_expected_breach,selected_coverage,selected_fitout,selected_economic_cost_36mo,best_cost_feasible,best_cost_feasible_value
0,Hyderabad,2047,2047,325,HYD-01|HYD-02|HYD-03|HYD-04|HYD-05|HYD-06|CS-07,1,1,70.959653,0.174910,0.767129,1580000.0,72.239530,HYD-01|HYD-02|HYD-03|HYD-04|HYD-05|HYD-06|CS-07,70.959653
1,Jaipur,1023,1023,153,JAI-01|JAI-02|JAI-03|JAI-04|JAI-05|CS-11,1,1,74.733965,0.199299,0.825131,1240000.0,76.522079,JAI-01|JAI-02|JAI-03|JAI-04|JAI-05|CS-11,74.733965
2,Pune,4095,4095,417,PUN-01|PUN-03|PUN-04|PUN-05|PUN-06|CS-01,1,1,66.335216,0.180191,0.829582,1650000.0,67.494864,PUN-01|PUN-03|PUN-04|PUN-05|PUN-06|CS-01,66.335216


### Static outputs
The exported charts are `images/demand_density.png`, `images/catchment_maps.png`, `images/breach_by_ring.png`, `images/before_after_impact.png`, and `images/optimization_frontier.png`. The full engineered order-level data is `order_level_cleaned.csv`; the counterfactual assignment is `after_order_assignment.csv`.